# FlyRank Search Intelligence Capstone: Master Pipeline Notebook
**FlyRank Search Intelligence Capstone Project**

Complete end-to-end executable capstone notebook: DuckDB ingestion, feature engineering, leak-free CV modeling, evaluation, and ranked content opportunity scoring engine.


In [ ]:
"""
===============================================================================
01_EDA_AND_DATA_PIPELINE.PY
Search Intelligence Capstone — FlyRank ML Internship Dataset Pipeline
===============================================================================
This module initializes the DuckDB analytical pipeline, loads/generates public-safe
search performance warehouse tables, computes foundational daily aggregations,
and fits an empirical position-to-CTR baseline curve.

Public Safety Rule Enforcement:
- No real domain names, URLs, credentials, or client identifiers.
- Anonymized identifiers (page_id_XXXX, query_cluster_YYY) are utilized throughout.
"""

import os
import sys
import numpy as np
import pandas as pd
import duckdb
from scipy.optimize import curve_fit
import json

def generate_flyrank_synthetic_warehouse(num_pages=500, days=90, seed=42):
    """
    Generates a realistic, anonymized search intelligence dataset in DuckDB format
    mimicking the FlyRank warehouse schema across a 90-day historical window.
    """
    np.random.seed(seed)
    start_date = pd.Timestamp("2026-05-28")
    date_range = [start_date + pd.Timedelta(days=i) for i in range(days)]
    
    records = []
    
    # Archetypes for pages:
    # 0: Stable high performers (20%)
    # 1: Decaying pages (Position drift downwards, traffic dropping) (30%)
    # 2: Underperforming CTR (High impressions, position 1-5, but CTR far below average) (25%)
    # 3: Growing / Momentum pages (25%)
    
    page_archetypes = np.random.choice([0, 1, 2, 3], size=num_pages, p=[0.20, 0.30, 0.25, 0.25])
    base_positions = np.random.uniform(1.2, 25.0, size=num_pages)
    base_impressions = np.random.exponential(scale=1500, size=num_pages) + 100
    
    for p_idx in range(num_pages):
        page_id = f"page_{p_idx+1000:04d}"
        archetype = page_archetypes[p_idx]
        base_pos = base_positions[p_idx]
        base_imp = base_impressions[p_idx]
        
        for t_idx, d in enumerate(date_range):
            # Time progress (0.0 to 1.0)
            t = t_idx / (days - 1)
            
            if archetype == 0:  # Stable
                pos_drift = np.random.normal(0, 0.2)
                imp_factor = np.random.normal(1.0, 0.05)
                ctr_factor = np.random.normal(1.0, 0.05)
            elif archetype == 1:  # Decaying
                pos_drift = t * np.random.uniform(3.0, 8.0) + np.random.normal(0, 0.3)
                imp_factor = max(0.2, 1.0 - 0.5 * t) + np.random.normal(0, 0.05)
                ctr_factor = max(0.3, 1.0 - 0.4 * t) + np.random.normal(0, 0.05)
            elif archetype == 2:  # CTR Underperformer
                pos_drift = np.random.normal(0, 0.3)
                imp_factor = np.random.normal(1.1, 0.05)
                ctr_factor = 0.35 + np.random.normal(0, 0.04) # low CTR relative to position
            else:  # Growing
                pos_drift = -t * np.random.uniform(2.0, 5.0) + np.random.normal(0, 0.3)
                imp_factor = (1.0 + 0.8 * t) + np.random.normal(0, 0.05)
                ctr_factor = np.random.normal(1.1, 0.05)
                
            curr_pos = max(1.0, base_pos + pos_drift)
            curr_imp = max(10, int(base_imp * imp_factor))
            
            # Expected CTR power-law formula: CTR ~ 0.30 / (pos ^ 1.1)
            expected_ctr = min(0.40, 0.32 / (curr_pos ** 1.12))
            actual_ctr = min(0.60, max(0.001, expected_ctr * ctr_factor))
            
            clicks = int(curr_imp * actual_ctr)
            query_count = max(1, int(np.sqrt(curr_imp) * np.random.uniform(0.8, 1.5)))
            
            records.append({
                "page_id": page_id,
                "date": d.strftime("%Y-%m-%d"),
                "impressions": curr_imp,
                "clicks": clicks,
                "ctr": round(clicks / max(1, curr_imp), 4),
                "position": round(curr_pos, 2),
                "query_count": query_count,
                "archetype_ground_truth": archetype
            })
            
    df = pd.DataFrame(records)
    return df

def run_duckdb_pipeline(data_dir="data"):
    """
    Initializes DuckDB connection, loads table, runs aggregate queries,
    and returns analytical summary.
    """
    os.makedirs(data_dir, exist_ok=True)
    df = generate_flyrank_synthetic_warehouse()
    
    db_path = os.path.join(data_dir, "search_intelligence.duckdb")
    con = duckdb.connect(db_path)
    
    # Register DataFrame in DuckDB
    con.register("df_raw", df)
    
    # Create persistent table in DuckDB
    con.execute("""
        CREATE OR REPLACE TABLE search_performance AS 
        SELECT 
            page_id,
            CAST(date AS DATE) as date,
            impressions,
            clicks,
            ctr,
            position,
            query_count,
            archetype_ground_truth
        FROM df_raw;
    """)
    
    print(f"[DuckDB] Ingested {con.execute('SELECT COUNT(*) FROM search_performance').fetchone()[0]} rows.")
    
    # Calculate global position vs CTR curve for baseline expectations
    pos_ctr_df = con.execute("""
        SELECT 
            ROUND(position, 0) as pos_bucket,
            COUNT(*) as sample_count,
            SUM(clicks) as total_clicks,
            SUM(impressions) as total_impressions,
            SUM(clicks) * 1.0 / NULLIF(SUM(impressions), 0) as empirical_ctr,
            AVG(position) as avg_exact_pos
        FROM search_performance
        WHERE position <= 20
        GROUP BY 1
        HAVING total_impressions > 500
        ORDER BY pos_bucket ASC;
    """).df()
    
    # Fit power law model: CTR = a * (pos ^ b)
    def power_law(x, a, b):
        return a * (x ** b)
    
    valid_data = pos_ctr_df.dropna()
    popt, _ = curve_fit(power_law, valid_data['avg_exact_pos'], valid_data['empirical_ctr'], p0=[0.3, -1.0])
    a_param, b_param = popt[0], popt[1]
    
    print(f"[Baseline Model] Fitted CTR expectation curve: Expected_CTR = {a_param:.4f} * (position ^ {b_param:.4f})")
    
    # Save baseline parameters
    baseline_params = {"a": a_param, "b": b_param}
    with open(os.path.join(data_dir, "ctr_baseline_params.json"), "w") as f:
        json.dump(baseline_params, f, indent=2)
        
    # Run Public Compliance & Privacy Audit
    print("\n--- Privacy & Public Safety Audit ---")
    columns = [row[0] for row in con.execute("DESCRIBE search_performance").fetchall()]
    sample = con.execute("SELECT * FROM search_performance LIMIT 5").df()
    print("Schema Columns:", columns)
    print("Sample Rows:\n", sample)
    
    # Check for PII / forbidden strings
    assert 'url' not in columns, "Privacy violation: raw URLs present in schema!"
    assert 'domain' not in columns, "Privacy violation: raw domains present in schema!"
    print("Privacy Audit PASSED: Zero private domains, URLs, or client identity present.")
    
    con.close()
    del con
    return db_path, baseline_params

if __name__ == "__main__":
    db_path, params = run_duckdb_pipeline()
    print("[Pipeline Complete] Data ingestion and EDA complete.")


# ==========================================
# STEP 2: FEATURE ENGINEERING
# ==========================================
"""
===============================================================================
02_FEATURE_ENGINEERING.PY
Search Intelligence Capstone — Feature Extraction & Label Definition
===============================================================================
This module extracts time-aware rolling window features from DuckDB and computes
search intelligence metrics: position decay slope, CTR deficit ratio, impression ratio,
and target opportunity labels (`needs_refresh`).
"""

import os
import json
import numpy as np
import pandas as pd
import duckdb

def compute_search_features(db_path="data/search_intelligence.duckdb", output_path="data/feature_matrix.parquet"):
    con = duckdb.connect(db_path, read_only=True)
    
    # Load baseline params
    params_path = "data/ctr_baseline_params.json"
    a_param, b_param = 0.32, -1.12
    if os.path.exists(params_path):
        with open(params_path, "r") as f:
            p = json.load(f)
            a_param, b_param = p["a"], p["b"]
            
    print(f"[Features] Loaded Baseline CTR Curve parameters: a={a_param:.4f}, b={b_param:.4f}")
    
    # Compute features via DuckDB SQL aggregations
    query = f"""
    WITH date_bounds AS (
        SELECT MAX(date) as max_date FROM search_performance
    ),
    aggregated AS (
        SELECT 
            sp.page_id,
            sp.archetype_ground_truth,
            
            -- Recent 7 days metrics (t-7 to t)
            AVG(CASE WHEN sp.date >= db.max_date - INTERVAL '7 days' THEN sp.position END) as pos_7d,
            SUM(CASE WHEN sp.date >= db.max_date - INTERVAL '7 days' THEN sp.impressions END) as imp_7d,
            SUM(CASE WHEN sp.date >= db.max_date - INTERVAL '7 days' THEN sp.clicks END) as clicks_7d,
            
            -- Mid 30 days metrics (t-30 to t)
            AVG(CASE WHEN sp.date >= db.max_date - INTERVAL '30 days' THEN sp.position END) as pos_30d,
            SUM(CASE WHEN sp.date >= db.max_date - INTERVAL '30 days' THEN sp.impressions END) as imp_30d,
            SUM(CASE WHEN sp.date >= db.max_date - INTERVAL '30 days' THEN sp.clicks END) as clicks_30d,
            
            -- Earlier baseline metrics (t-90 to t-30)
            AVG(CASE WHEN sp.date < db.max_date - INTERVAL '30 days' THEN sp.position END) as pos_historical,
            SUM(CASE WHEN sp.date < db.max_date - INTERVAL '30 days' THEN sp.impressions END) as imp_historical,
            SUM(CASE WHEN sp.date < db.max_date - INTERVAL '30 days' THEN sp.clicks END) as clicks_historical,
            
            -- Overall window metrics
            AVG(sp.position) as pos_90d,
            SUM(sp.impressions) as imp_90d,
            SUM(sp.clicks) as clicks_90d,
            STDDEV(sp.position) as pos_volatility_std,
            AVG(sp.query_count) as avg_query_count
            
        FROM search_performance sp, date_bounds db
        GROUP BY sp.page_id, sp.archetype_ground_truth
    )
    SELECT 
        page_id,
        archetype_ground_truth,
        
        pos_7d,
        pos_30d,
        pos_historical,
        pos_90d,
        
        imp_7d,
        imp_30d,
        imp_historical,
        imp_90d,
        
        clicks_7d,
        clicks_30d,
        clicks_historical,
        clicks_90d,
        
        -- Engineered Features
        (pos_30d - pos_historical) as pos_drift_30d_vs_hist,
        (pos_7d - pos_30d) as pos_drift_7d_vs_30d,
        
        (imp_30d * 1.0 / NULLIF(imp_historical / 2.0, 0)) as imp_ratio_30d_vs_hist,
        
        (clicks_30d * 1.0 / NULLIF(imp_30d, 0)) as ctr_observed_30d,
        
        pos_volatility_std,
        avg_query_count
        
    FROM aggregated;
    """
    
    df_feat = con.execute(query).df()
    con.close()
    
    # Vectorized post-processing in pandas/numpy
    # 1. Compute expected CTR based on empirical baseline curve
    df_feat['ctr_expected_30d'] = a_param * (df_feat['pos_30d'].clip(lower=1.0) ** b_param)
    df_feat['ctr_expected_30d'] = df_feat['ctr_expected_30d'].clip(upper=0.40)
    
    # 2. Compute CTR Deficit Ratio: (Expected - Observed) / Expected
    df_feat['ctr_deficit_ratio'] = (df_feat['ctr_expected_30d'] - df_feat['ctr_observed_30d']) / np.maximum(0.001, df_feat['ctr_expected_30d'])
    df_feat['ctr_deficit_ratio'] = df_feat['ctr_deficit_ratio'].clip(-1.0, 2.0)
    
    # 3. Traffic / Click Decay Velocity
    df_feat['click_decay_velocity'] = (df_feat['clicks_30d'] / 30.0) - (df_feat['clicks_historical'] / 60.0)
    
    # 4. Target Label definition (Needs Refresh Opportunity):
    # High opportunity if:
    # (a) Position drift >= 1.5 rank drop OR
    # (b) CTR Deficit Ratio >= 0.35 (observed CTR is 35%+ below baseline expectation for its position) OR
    # (c) Ground truth archetype is Decaying (1) or CTR Underperformer (2)
    df_feat['target_needs_refresh'] = np.where(
        (df_feat['pos_drift_30d_vs_hist'] >= 1.2) | 
        (df_feat['ctr_deficit_ratio'] >= 0.30) |
        (df_feat['archetype_ground_truth'].isin([1, 2])), 
        1, 0
    )
    
    # Clean NaNs/Infs
    df_feat.fillna(0, inplace=True)
    
    # Save to Parquet
    df_feat.to_parquet(output_path, index=False)
    print(f"[Features] Saved {len(df_feat)} engineered page feature vectors to {output_path}")
    print(f"[Label Balance] Needs Refresh: {df_feat['target_needs_refresh'].sum()} / {len(df_feat)} ({df_feat['target_needs_refresh'].mean()*100:.1f}%)")
    
    return df_feat

if __name__ == "__main__":
    compute_search_features()


# ==========================================
# STEP 3: MODELING & CV EVALUATION
# ==========================================
"""
===============================================================================
03_MODEL_TRAINING_AND_EVALUATION.PY
Search Intelligence Capstone — Model Training, Cross-Validation & Evaluation
===============================================================================
This module evaluates ML classifiers against rule-based baselines using leak-free
cross-validation. It outputs quantitative performance metrics, confusion matrices,
and feature importance rankings.
"""

import os
import json
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score, precision_recall_curve, auc, f1_score, precision_score, recall_score, confusion_matrix

def evaluate_models(feature_path="data/feature_matrix.parquet", output_dir="data"):
    os.makedirs(output_dir, exist_ok=True)
    df = pd.read_parquet(feature_path)
    
    feature_cols = [
        'pos_7d', 'pos_30d', 'pos_historical', 'pos_90d',
        'imp_7d', 'imp_30d', 'imp_historical', 'imp_90d',
        'clicks_7d', 'clicks_30d', 'clicks_90d',
        'pos_drift_30d_vs_hist', 'pos_drift_7d_vs_30d',
        'imp_ratio_30d_vs_hist', 'ctr_observed_30d',
        'ctr_expected_30d', 'ctr_deficit_ratio',
        'click_decay_velocity', 'pos_volatility_std', 'avg_query_count'
    ]
    
    X = df[feature_cols]
    y = df['target_needs_refresh']
    
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    
    models = {
        "Rule_Based_Baseline": "rule",
        "Logistic_Regression": Pipeline([('scaler', StandardScaler()), ('clf', LogisticRegression(random_state=42))]),
        "Random_Forest": RandomForestClassifier(n_estimators=100, max_depth=8, random_state=42),
        "Hist_Gradient_Boosting": HistGradientBoostingClassifier(max_iter=100, max_depth=5, random_state=42)
    }
    
    results = {}
    
    for name, model in models.items():
        print(f"\n[Evaluating] {name}...")
        
        y_true_all = []
        y_pred_all = []
        y_prob_all = []
        
        for train_idx, val_idx in skf.split(X, y):
            X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
            y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
            
            if name == "Rule_Based_Baseline":
                # Rule heuristic: position drift >= 1.0 or CTR deficit >= 0.25
                preds = ((X_val['pos_drift_30d_vs_hist'] >= 1.0) | (X_val['ctr_deficit_ratio'] >= 0.25)).astype(int)
                probs = np.where(X_val['ctr_deficit_ratio'] > 0, X_val['ctr_deficit_ratio'].clip(0, 1), 0.1)
            else:
                model.fit(X_train, y_train)
                preds = model.predict(X_val)
                probs = model.predict_proba(X_val)[:, 1]
                
            y_true_all.extend(y_val)
            y_pred_all.extend(preds)
            y_prob_all.extend(probs)
            
        y_true_all = np.array(y_true_all)
        y_pred_all = np.array(y_pred_all)
        y_prob_all = np.array(y_prob_all)
        
        # Calculate evaluation metrics
        roc_val = roc_auc_score(y_true_all, y_prob_all)
        p, r, _ = precision_recall_curve(y_true_all, y_prob_all)
        pr_auc_val = auc(r, p)
        f1_val = f1_score(y_true_all, y_pred_all)
        prec_val = precision_score(y_true_all, y_pred_all)
        rec_val = recall_score(y_true_all, y_pred_all)
        cm = confusion_matrix(y_true_all, y_pred_all).tolist()
        
        results[name] = {
            "ROC_AUC": round(float(roc_val), 4),
            "PR_AUC": round(float(pr_auc_val), 4),
            "F1_Score": round(float(f1_val), 4),
            "Precision": round(float(prec_val), 4),
            "Recall": round(float(rec_val), 4),
            "Confusion_Matrix": cm
        }
        
        print(f"   ROC-AUC: {roc_val:.4f} | PR-AUC: {pr_auc_val:.4f} | F1: {f1_val:.4f} | Precision: {prec_val:.4f} | Recall: {rec_val:.4f}")
        
    # Fit final champion model (HistGradientBoosting) on full dataset to get feature importances & scoring predictions
    final_model = HistGradientBoostingClassifier(max_iter=100, max_depth=5, random_state=42)
    final_model.fit(X, y)
    
    rf_model = RandomForestClassifier(n_estimators=100, max_depth=8, random_state=42)
    rf_model.fit(X, y)
    
    importances = dict(zip(feature_cols, rf_model.feature_importances_))
    sorted_importances = dict(sorted(importances.items(), key=lambda item: item[1], reverse=True))
    
    # Save evaluation outputs to JSON
    with open(os.path.join(output_dir, "evaluation_metrics.json"), "w") as f:
        json.dump(results, f, indent=2)
        
    with open(os.path.join(output_dir, "feature_importances.json"), "w") as f:
        json.dump(sorted_importances, f, indent=2)
        
    print(f"\n[Model Training Complete] Saved evaluation metrics & feature importances to {output_dir}")
    return results, sorted_importances

if __name__ == "__main__":
    evaluate_models()


# ==========================================
# STEP 4: OPPORTUNITY SCORING ENGINE
# ==========================================
"""
===============================================================================
04_CAPSTONE_OPPORTUNITY_SCORING.PY
Search Intelligence Capstone — Content Opportunity Scoring Engine
===============================================================================
This module applies the trained champion ML model to generate continuous
opportunity scores (0-100), automated diagnostic reason codes, and actionable
content playbooks. Results are exported to data/ranked_recommendations.json.
"""

import os
import json
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier

def generate_opportunity_engine(feature_path="data/feature_matrix.parquet", output_json="data/ranked_recommendations.json"):
    df = pd.read_parquet(feature_path)
    
    feature_cols = [
        'pos_7d', 'pos_30d', 'pos_historical', 'pos_90d',
        'imp_7d', 'imp_30d', 'imp_historical', 'imp_90d',
        'clicks_7d', 'clicks_30d', 'clicks_90d',
        'pos_drift_30d_vs_hist', 'pos_drift_7d_vs_30d',
        'imp_ratio_30d_vs_hist', 'ctr_observed_30d',
        'ctr_expected_30d', 'ctr_deficit_ratio',
        'click_decay_velocity', 'pos_volatility_std', 'avg_query_count'
    ]
    
    X = df[feature_cols]
    y = df['target_needs_refresh']
    
    # Train final production model
    clf = HistGradientBoostingClassifier(max_iter=100, max_depth=5, random_state=42)
    clf.fit(X, y)
    
    # Calculate refresh opportunity probabilities (0 to 100)
    probs = clf.predict_proba(X)[:, 1]
    df['opportunity_score'] = (probs * 100).round(1)
    
    # Assign Diagnostic Reason Codes & Action Recommendations
    recommendations = []
    
    for idx, row in df.iterrows():
        score = row['opportunity_score']
        pos_drift = row['pos_drift_30d_vs_hist']
        ctr_deficit = row['ctr_deficit_ratio']
        pos_30d = row['pos_30d']
        imp_30d = row['imp_30d']
        
        reason_codes = []
        
        if pos_drift >= 1.5:
            reason_codes.append("DECAY_POSITION_SLIP")
        if ctr_deficit >= 0.35:
            reason_codes.append("CTR_UNDERPERFORMING")
        if imp_30d > 2500 and ctr_deficit >= 0.20:
            reason_codes.append("HIGH_IMP_LOW_CLICK")
        if row['pos_volatility_std'] > 2.5:
            reason_codes.append("MONITOR_VOLATILITY")
            
        if not reason_codes:
            if score < 30.0:
                reason_codes.append("STABLE_PERFORMER")
            else:
                reason_codes.append("MODERATE_DECAY_RISK")
                
        # Primary Action Recommendation
        if "DECAY_POSITION_SLIP" in reason_codes and "CTR_UNDERPERFORMING" in reason_codes:
            action = "REWRITE_AND_UPDATE_METADATA"
            urgency = "CRITICAL"
        elif "DECAY_POSITION_SLIP" in reason_codes:
            action = "REWRITE_CONTENT_INTENT"
            urgency = "HIGH"
        elif "CTR_UNDERPERFORMING" in reason_codes or "HIGH_IMP_LOW_CLICK" in reason_codes:
            action = "OPTIMIZE_METADATA_TITLES"
            urgency = "HIGH"
        elif "MONITOR_VOLATILITY" in reason_codes:
            action = "MONITOR_QUERY_INTENT"
            urgency = "MEDIUM"
        else:
            action = "PROTECT_AND_MONITOR"
            urgency = "LOW"
            
        recommendations.append({
            "rank": 0, # To be sorted
            "page_id": row['page_id'],
            "opportunity_score": float(score),
            "pos_30d": round(float(pos_30d), 1),
            "pos_drift": round(float(pos_drift), 2),
            "imp_30d": int(imp_30d),
            "clicks_30d": int(row['clicks_30d']),
            "ctr_observed": round(float(row['ctr_observed_30d']) * 100, 2),
            "ctr_expected": round(float(row['ctr_expected_30d']) * 100, 2),
            "ctr_deficit_pct": round(float(ctr_deficit) * 100, 1),
            "reason_codes": reason_codes,
            "recommended_action": action,
            "urgency": urgency
        })
        
    # Sort recommendations by opportunity score descending
    recommendations.sort(key=lambda x: x['opportunity_score'], reverse=True)
    for r_idx, item in enumerate(recommendations):
        item['rank'] = r_idx + 1
        
    with open(output_json, "w") as f:
        json.dump(recommendations, f, indent=2)
        
    print(f"[Scoring Engine] Generated {len(recommendations)} ranked page recommendations in {output_json}")
    
    # Print Top 5 Opportunity Pages
    print("\n--- TOP 5 CONTENT REFRESH OPPORTUNITY PAGES ---")
    for item in recommendations[:5]:
        print(f"Rank #{item['rank']} | {item['page_id']} | Score: {item['opportunity_score']}/100 | Action: {item['recommended_action']} | Reasons: {', '.join(item['reason_codes'])}")
        
    return recommendations

if __name__ == "__main__":
    generate_opportunity_engine()
